# Diagnóstico: renda, NDVI e temperatura da superfície terrestre

Este notebook verifica se o padrão inverso entre renda e NDVI pode ser explicado pela validade dos dados, pelas flags dos setores ou pelo pré-processamento Landsat. Ele não altera nenhum arquivo processado existente.

Este notebook verifica a validade da renda, a composição espacial dos setores, o pré-processamento Landsat e a sensibilidade dos resultados às flags territoriais. Ele é uma auditoria, não uma nova etapa de produção.

**Entradas:** tabelas processadas de renda, NDVI e temperatura. **Saídas:** tabelas e gráficos de diagnóstico.


In [ ]:
# O quintil só é interpretado para setores com renda válida; ausências permanecem ausências.

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"
income = pd.read_csv(PROCESSED / "sectors_income.csv")
metrics = pd.read_csv(PROCESSED / "sector_lst_ndvi.csv")
income["sector_id"] = income["sector_id"].astype("string").str.zfill(15)
metrics["sector_id"] = metrics["sector_id"].astype("string").str.zfill(15)
analysis = income.merge(metrics, on="sector_id", how="left", validate="one_to_one")
analysis["income_quintile"] = analysis["income_quintile"].astype("Int64")
analysis.shape

## 1. Validade da renda e escopo dos setores

As variáveis de renda são valores mensais nominais em reais, com julho de 2022 como mês de referência. Os quintis devem ser definidos somente para setores com renda mediana válida.

O objetivo é verificar ausências e definir claramente o universo de setores. A análise conserva as flags para que exclusões sejam visíveis.


In [ ]:
print("Rows:", len(analysis))
print("Null median income:", analysis["income_median"].isna().sum())
print("Zero median income:", (analysis["income_median"] == 0).sum())
print("Valid income:", analysis["has_income"].sum())
print("Non-null quintile among valid income:", analysis.loc[analysis["has_income"], "income_quintile"].notna().sum())
print("Valid-income rural sectors:", int(analysis.loc[analysis["has_income"], "is_rural"].sum()))
print("Valid-income small sectors:", int(analysis.loc[analysis["has_income"], "small_sector"].sum()))
print("Valid income, non-rural, non-small:", int((analysis["has_income"] & ~analysis["is_rural"] & ~analysis["small_sector"]).sum()))
print("Median income range (BRL/month):", analysis["income_median"].min(), "to", analysis["income_median"].max())
print("Median income unit: Brazilian reais per month, July 2022")

## 2. Área dos setores por quintil de renda

O objetivo é comparar a distribuição de área entre quintis e reconhecer setores muito pequenos. TODO: justificar qualquer uso substantivo de um corte de área além da flag já criada.


In [ ]:
area_by_quintile = (
    analysis.loc[analysis["has_income"]]
    .groupby("income_quintile")["area_km2"]
    .agg(n_sectors="size", mean_area_km2="mean", median_area_km2="median")
)
area_by_quintile

## 3. Auditoria do pré-processamento Landsat

O notebook original do Earth Engine aplica os fatores de escala do Collection 2 Level 2 e mascara fill, nuvem dilatada, cirrus, nuvem, sombra de nuvem e água por meio de `QA_PIXEL`. A conversão de temperatura também transforma Kelvin em Celsius.

O objetivo é auditar máscaras e fatores de escala do Landsat. A decisão documentada no código é reproduzir a lógica de `QA_PIXEL` e conversão para Celsius.


In [ ]:
preprocessing_code = '''
qa = image.select("QA_PIXEL")
excluded_qa_bits = (1 << 0) | (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4) | (1 << 7)
clear_land = qa.bitwiseAnd(excluded_qa_bits).eq(0)
red = image.select("SR_B4").multiply(0.0000275).add(-0.2)
nir = image.select("SR_B5").multiply(0.0000275).add(-0.2)
lst_c = image.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15)
ndvi = nir.subtract(red).divide(nir.add(red)).updateMask(clear_land)
lst_c = lst_c.updateMask(clear_land)
'''
print(preprocessing_code)
print("SR scale: stored value * 0.0000275 - 0.2")
print("ST scale: stored value * 0.00341802 + 149.0, then Kelvin - 273.15")
print("QA_PIXEL bits: 0 fill, 1 dilated cloud, 2 cirrus, 3 cloud, 4 cloud shadow, 7 water")

## 4. Resumos por quintil e sensibilidade às flags

O objetivo é comparar resultados em escopos diferentes, incluindo setores rurais e pequenos quando apropriado. Isso mostra a sensibilidade das conclusões às flags.


In [ ]:
# Os escopos separam o resultado geral das verificações com flags territoriais.

scopes = {
    "valid_income": analysis["has_income"],
    "valid_income_nonrural": analysis["has_income"] & ~analysis["is_rural"],
    "valid_income_nonrural_nonsmall": analysis["has_income"] & ~analysis["is_rural"] & ~analysis["small_sector"],
}

summary_tables = {}
correlation_tables = {}
for scope_name, scope in scopes.items():
    scoped = analysis.loc[scope].copy()
    summary_tables[scope_name] = scoped.groupby("income_quintile").agg(
        n_sectors=("sector_id", "size"),
        n_valid_ndvi=("ndvi", "count"),
        n_valid_lst=("lst_c", "count"),
        mean_ndvi=("ndvi", "mean"),
        median_ndvi=("ndvi", "median"),
        mean_lst_c=("lst_c", "mean"),
        median_lst_c=("lst_c", "median"),
    )
    complete = scoped.dropna(subset=["income_median", "ndvi", "lst_c"])
    correlation_tables[scope_name] = pd.Series({
        "income_vs_ndvi_rho": complete["income_median"].corr(complete["ndvi"], method="spearman"),
        "income_vs_lst_rho": complete["income_median"].corr(complete["lst_c"], method="spearman"),
        "ndvi_vs_lst_rho": complete["ndvi"].corr(complete["lst_c"], method="spearman"),
        "complete_cases": len(complete),
    })
summary_tables["valid_income"], pd.DataFrame(correlation_tables).T

In [ ]:
plot_data = analysis.loc[analysis["has_income"]].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
plot_data.boxplot(column="ndvi", by="income_quintile", ax=axes[0], grid=False)
plot_data.boxplot(column="lst_c", by="income_quintile", ax=axes[1], grid=False)
axes[0].set_title("NDVI by income quintile")
axes[1].set_title("Land surface temperature by income quintile")
for ax in axes:
    ax.set_xlabel("Income quintile")
fig.suptitle("")
plt.show()


## 5. Setores com menor NDVI no quintil mais rico

O objetivo é inspecionar casos de NDVI baixo no quintil de maior renda sem transformar esses casos em uma regra geral.


In [ ]:
richest_low_ndvi = (
    analysis.loc[analysis["has_income"] & (analysis["income_quintile"] == 5)]
    .nsmallest(10, "ndvi")
    [["sector_id", "income_median", "income_mean", "ndvi", "lst_c", "area_km2", "is_rural", "small_sector", "in_rural_zone", "sector_type", "situation_code", "subdistrict"]]
)
richest_low_ndvi

## Conclusão da etapa

O diagnóstico torna explícitos os casos ausentes e mostra como as comparações mudam quando flags de renda, ruralidade e tamanho são consideradas. Ele serve para qualificar a interpretação das análises principais.
